# 14 — Benamou–Brenier and Otto

Close Module 3 by viewing the $W_2$ geodesic two ways — McCann's explicit affine map, and Benamou–Brenier's least-energy flow — and complete the dictionary that carries transport into the quantum world.

**Prerequisites:** `12_bures_wasserstein`, `13_the_bures_bridge`, `07_mccann_geodesic`.

**What you'll be able to do**
- Build the affine McCann map between Gaussians and trace the $W_2$ geodesic (ellipses that translate, rotate, rescale).
- State the Benamou–Brenier dynamic form and Otto's Riemannian view of Wasserstein space.
- Complete the classical↔quantum dictionary and close Module 3.

One arc remains, and it closes Module 3. You have a distance on Gaussians (12) and the bridge to quantum states (13). Now meet the *geodesic* — the optimal path — in two complementary readings: McCann's explicit affine map that morphs one ellipse into another, and Benamou and Brenier's picture of mass flowing with least kinetic energy. Together they make Wasserstein space a curved geometry and complete the dictionary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from qot_course import viz
from qot_course.transport.gaussian import gaussian_ot_map, gaussian_geodesic

np.random.seed(42)
viz.use_course_style()

## Where we are

Two bricks brought us here: `12_bures_wasserstein` gave the closed-form $W_2$ between Gaussians, and `13_the_bures_bridge` showed its matrix term *is* the quantum Bures distance. Now we trace the geodesic between two Gaussians — first as an explicit map, then as a flow — and complete the dictionary before Module 4 lifts it all to density matrices.

## 1. The $W_2$ geodesic — an affine McCann map

Between Gaussians the optimal transport map is affine and explicit. For zero-mean covariances,

$$ T(x) = A\,x, \qquad A = \Sigma_0^{-1/2}\bigl(\Sigma_0^{1/2}\,\Sigma_1\,\Sigma_0^{1/2}\bigr)^{1/2}\Sigma_0^{-1/2}, $$

the unique symmetric positive-definite $A$ with $A\Sigma_0 A = \Sigma_1$; for general means $T(x) = m_1 + A(x - m_0)$. The $W_2$ geodesic therefore stays Gaussian,

$$ \mu_t = \mathcal{N}\big((1-t)m_0 + t\,m_1,\ M_t\,\Sigma_0\,M_t^\top\big), \qquad M_t = (1-t)I + tA, $$

rotating, translating, and rescaling the covariance ellipse all at once. Let's draw it.

In [ ]:
mean_0 = np.array([-2.5, -1.0])
cov_0  = np.array([[1.5, 0.4], [0.4, 0.6]])
mean_1 = np.array([2.5, 1.5])
cov_1  = np.array([[0.5, -0.3], [-0.3, 1.4]])

def ellipse(mean, cov, n_std=1.5, **kwargs):
    """A covariance ellipse at n_std standard deviations."""
    vals, vecs = np.linalg.eigh(0.5 * (cov + cov.T))
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = float(np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0])))
    width, height = (2.0 * n_std * np.sqrt(np.clip(vals, 0.0, None))).tolist()
    return patches.Ellipse(xy=tuple(mean), width=width, height=height, angle=angle, **kwargs)

A = gaussian_ot_map(cov_0, cov_1)
print(f"A Sigma_0 A == Sigma_1 ?  {bool(np.allclose(A @ cov_0 @ A.T, cov_1, atol=1e-9))}")
print(f"A symmetric (SPD)?        {bool(np.allclose(A, A.T) and np.all(np.linalg.eigvalsh(A) > 0))}")

ts = np.linspace(0.0, 1.0, 9)
snapshots = [gaussian_geodesic(mean_0, cov_0, mean_1, cov_1, t) for t in ts]
means = np.array([s[0] for s in snapshots])

fig, ax = plt.subplots(figsize=(10, 6))
cmap = plt.colormaps["viridis"]
for k, (t, (mean_t, cov_t)) in enumerate(zip(ts, snapshots)):
    shade = k / (len(ts) - 1)
    ax.add_patch(ellipse(mean_t, cov_t,
                         edgecolor=cmap(shade), facecolor=cmap(shade),
                         alpha=0.18 + 0.5 * shade, linewidth=2.0,
                         label=f"t = {t:.2f}" if k in (0, len(ts) // 2, len(ts) - 1) else None))
ax.plot(means[:, 0], means[:, 1], "--", color=viz.COLORS["text"], lw=1.2, label="mean trajectory")
ax.scatter(means[[0, -1], 0], means[[0, -1], 1], color=viz.COLORS["text"], s=70, zorder=4)
ax.set_xlim(-6, 6)
ax.set_ylim(-5, 5)
ax.set_aspect("equal")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("W2 geodesic in the Gaussian family: translate + rotate + rescale at once", pad=12)
ax.legend(loc="upper left")
plt.show()

**Read the figure.** The map $A$ pushes $\Sigma_0$ exactly onto $\Sigma_1$ (verified above). The geodesic ellipses flow from the lower-left Gaussian (violet) to the upper-right one (yellow): the mean tracks a straight line, while the covariance shape blends so that the distribution **stays Gaussian at every $t$**. That closed-form geodesic is a luxury we have only because the family is Gaussian — and it is exactly the displacement-interpolation idea of `07_mccann_geodesic`, now in two dimensions and in matrix form.

## 2. The dynamic view — Benamou–Brenier and Otto

The static problem has a fluid-dynamics twin (Benamou & Brenier, 2000):

$$ W_2^2(\mu_0, \mu_1) = \inf_{(\rho_t, v_t)}\ \int_0^1\!\!\int |v_t(x)|^2\,\rho_t(x)\,\mathrm{d}x\,\mathrm{d}t, $$

minimised over a path of distributions $\rho_t$ and a velocity field $v_t$ obeying the **continuity equation** $\partial_t \rho_t + \nabla\!\cdot(\rho_t v_t) = 0$ with $\rho_0 = \mu_0$, $\rho_1 = \mu_1$. In words: move the mass from $\mu_0$ to $\mu_1$ with the least total kinetic energy. The optimal velocity is the gradient of a Kantorovich potential, and the optimal $\rho_t$ is the McCann interpolation of `07`.

**Otto (2001)** read this as geometry: it gives the space of distributions $\mathcal{P}_2$ a **Riemannian-like metric**, with velocity fields as tangent vectors and McCann interpolations as geodesics. So Wasserstein space is *curved* — like the Fisher–Rao geometry of `02/06`, but one that respects the ground space rather than only the amplitudes. In Module 4 the continuity equation becomes a Lindblad-type evolution on density matrices and Otto's metric becomes a quantum Riemannian metric — the bridge holds one more time.

## 3. Dictionary update — the Bures bridge, complete

The Gaussian arc completes the classical↔quantum dictionary. Every row now has a quantum counterpart waiting in Module 4.

| Classical | Quantum |
|-----------|---------|
| Gaussian $\mathcal{N}(m, \Sigma)$ on $\mathbb{R}^d$ | Gaussian state on the oscillator  *(Module 4)* |
| **covariance matrix $\Sigma$ (PSD, finite trace)** | **density matrix $\rho$ (PSD, unit trace)** |
| **$W_2$ closed form** $\|\Delta m\|^2 + \mathrm{tr}\,\Sigma_0 + \mathrm{tr}\,\Sigma_1 - 2\,\mathrm{tr}\sqrt{\Sigma_0^{1/2}\Sigma_1\Sigma_0^{1/2}}$ | **Bures distance** $d_B^2 = 2(1 - F_U)$ — the same formula on unit-trace PSD |
| **McCann map** $T(x) = m_1 + A(x - m_0)$ | **quantum coupling map**  *(Module 4)* |
| **$W_2$ geodesic** on Gaussians | **Bures geodesic** on density matrices (implicit in `02/12` / Uhlmann) |
| Benamou–Brenier dynamic form | **quantum continuity equation**  *(Module 4)* |
| Otto calculus on $\mathcal{P}_2$ | **quantum Riemannian metrics on $\mathcal{S}(\mathcal{H})$**  *(Module 4; Carlen–Maas)* |

Module 3 is complete. The next module lifts each row to density matrices.

## Your turn

1. **Eigenvectors of $A$.** Take diagonal covariances $\Sigma_0 = \mathrm{diag}(1, 4)$ and $\Sigma_1 = \mathrm{diag}(9, 1)$. Predict the eigenvalues of the OT map $A$ (hint: $\sqrt{\sigma_1/\sigma_0}$ in each axis) and check with `gaussian_ot_map`.
2. **Constant-speed flow.** Along the Gaussian geodesic, confirm $W_2(\mu_0, \mu_t)$ grows linearly in $t$ — a geodesic has constant speed, exactly as in `07`.
3. **Kinetic energy.** For the 1-D geodesic between $\mathcal{N}(0, 1)$ and $\mathcal{N}(4, 1)$ (a pure translation), argue that the Benamou–Brenier velocity field is constant and that the total kinetic energy equals $W_2^2$.

## What you built

- You built the affine McCann map between Gaussians and traced the $W_2$ geodesic — ellipses translating, rotating, and rescaling, staying Gaussian throughout.
- You met the Benamou–Brenier dynamic form and Otto's Riemannian reading of Wasserstein space.
- You completed the classical↔quantum dictionary.

**Module 3 is complete.** From Monge's deterministic map, through the Kantorovich LP, the Wasserstein metric and its geodesics, the Sinkhorn algorithm and Amari's bridge, to the Gaussian closed form and the Bures bridge — you have built the whole of classical optimal transport, and the dictionary that lets it speak quantum. That is a substantial body of understanding, and it is the launchpad for everything ahead.

## What's next

Module 4 — **quantum optimal transport** — lifts every row of the dictionary to density matrices. It opens by asking *why* a genuinely quantum transport is needed at all: what the naive "diagonal" approach misses, and how the $|+\rangle$-vs-mixed lesson and the Bures bridge you built point the way to quantum Wasserstein.

## References

- J.-D. Benamou & Y. Brenier, "A computational fluid mechanics solution to the Monge–Kantorovich mass transfer problem", *Numer. Math.* 84, 375–393 (2000).
- F. Otto, "The geometry of dissipative evolution equations: the porous medium equation", *Comm. PDE* 26, 101–174 (2001).
- E. A. Carlen & J. Maas, "An analog of the 2-Wasserstein metric in non-commutative probability...", *Comm. Math. Phys.* 331, 887–926 (2014).
- R. McCann, "A convexity principle for interacting gases", *Adv. Math.* 128, 153–179 (1997).

**Previous:** `notebooks/03_ClassicalOptimalTransport/13_the_bures_bridge.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/01_why_qot.ipynb`